## Persiapan

In [ ]:
import os, zipfile, urllib.request
# Tautan unduhan data dari repo GitHub
URL_DATA = "https://raw.githubusercontent.com/Febrianggi/praktikum-geospasial-fokasi/main/data_praktikum.zip"
if not os.path.isdir("data") and not os.path.exists("data_praktikum.zip") and "NAMA_PENGGUNA" not in URL_DATA:
    urllib.request.urlretrieve(URL_DATA, "data_praktikum.zip")
if not os.path.isdir("data") and os.path.exists("data_praktikum.zip"):
    zipfile.ZipFile("data_praktikum.zip").extractall(".")
assert os.path.isdir("data"), "Folder data/ belum ada: unggah data_praktikum.zip ke panel Files Colab"
try:
    import mapclassify, folium, libpysal, esda  # noqa
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "mapclassify", "folium", "libpysal", "esda"], check=True)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import folium
from shapely.validation import make_valid
from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local

os.makedirs("output", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def isi(nilai, contoh, petunjuk):
    if nilai is None:
        print(f"⚠ Belum diisi ({petunjuk}). Memakai jawaban contoh: {contoh!r}")
        return contoh
    return nilai
print("Siap. geopandas", gpd.__version__)

## Batas 38 provinsi, tabel BPS, sistem koordinat
Data spasial = **atribut** (angka BPS) + **geometri** (poligon provinsi).

**Catatan sumber batas wilayah:** GeoJSON 38 provinsi dari repositori publik `denyherianto/indonesia-geojson-topojson-maps-with-38-provinces` (GitHub).

In [ ]:
prov_mentah = gpd.read_file("data/sumber/batas_38_provinsi_denyherianto.geojson")
print(len(prov_mentah), "poligon | CRS:", prov_mentah.crs)
print("Geometri tidak valid:", int((~prov_mentah.geometry.is_valid).sum()))
prov_mentah["geometry"] = prov_mentah.geometry.apply(make_valid)
prov_mentah[["PROVINSI", "KODE_PROV"]].head()

In [ ]:
bps = pd.read_csv("data/olahan/bps_tabel_2_5_14_ketahanan_bangunan.csv")
print(bps.shape[0], "baris provinsi")
print("Sumber:", bps["sumber"].iloc[0])
bps[["kode_bps", "provinsi_bps", "perkotaan_2024", "perdesaan_2024", "gabungan_2024"]].head()

In [ ]:
# Derajat (EPSG:4326) untuk menyimpan; meter (EPSG:6933, equal-area) untuk luas dan jarak
prov_m = prov_mentah.to_crs("EPSG:6933")
prov_mentah["luas_km2"] = (prov_m.area / 1e6).round(0)
prov_mentah.sort_values("luas_km2", ascending=False)[["PROVINSI", "luas_km2"]].head(3)

## Mencatat koordinat GPS dan akurasinya

Templat kosong ada di `data/templat_pencatatan_gps.csv`. Fungsi validasi di bawah dipakai untuk data survei apa pun. Di sini fungsi itu dicoba pada katalog USGS, yang juga mencatat akurasi lokasi (`horizontalError`, km).

In [ ]:
templat = pd.read_csv("data/templat_pencatatan_gps.csv")
print("Kolom templat:", list(templat.columns))

def validasi_koordinat(df, lat="latitude", lon="longitude", akurasi=None):
    """Kembalikan alasan gagal per baris ('' = lolos). Tidak mengubah atau menebak nilai."""
    la = pd.to_numeric(df[lat], errors="coerce"); lo = pd.to_numeric(df[lon], errors="coerce")
    alasan = pd.Series("", index=df.index)
    alasan[la.isna() | lo.isna()] = "koordinat kosong atau bukan angka"
    alasan[(alasan == "") & (~la.between(-90, 90) | ~lo.between(-180, 180))] = "di luar rentang bumi"
    alasan[(alasan == "") & (la.abs() > lo.abs()) & (lo.abs() < 15)] = "lintang/bujur kemungkinan tertukar"
    if akurasi:
        alasan[(alasan == "") & df[akurasi].isna()] = "akurasi tidak tercatat"
    return alasan

In [ ]:
usgs = pd.read_csv("data/sumber/usgs_gempa_2026_M4.5.csv")
cek = validasi_koordinat(usgs, akurasi="horizontalError")
print("Lolos:", (cek == "").sum(), "dari", len(usgs))
print("Akurasi horizontal USGS (km):", usgs["horizontalError"].describe()[["min", "50%", "max"]].round(1).to_dict())

Pertanyaan: median ketidakpastian lokasi episentrum sekitar 9 km. Apa artinya bila sebuah episentrum tampak tepat di garis pantai?

**Jawaban:** Posisi episentrum punya ketidakpastian beberapa kilometer, jadi titik di garis pantai bisa saja sebenarnya di laut atau di darat. Karena itu peta episentrum dibaca sebagai pola, bukan lokasi pasti, dan kita tidak menggeser titik ke provinsi terdekat.

## Menggabung tabel ke poligon secara 1:1
Kunci gabung: **kode BPS**. Kita buat kode dari nama provinsi (crosswalk), lalu `merge(..., validate="one_to_one")`. Satu nama berbeda ejaan: *Daerah Istimewa Yogyakarta* (peta) vs *DI Yogyakarta* (BPS).

In [ ]:
KODE = dict(zip(bps["provinsi_bps"], bps["kode_bps"]))
ejaan = {"Daerah Istimewa Yogyakarta": "DI Yogyakarta"}
prov_mentah["provinsi_bps"] = prov_mentah["PROVINSI"].replace(ejaan)
prov_mentah["kode_bps"] = prov_mentah["provinsi_bps"].map(KODE)
print("Nama peta tanpa kode BPS:", prov_mentah.loc[prov_mentah["kode_bps"].isna(), "PROVINSI"].tolist())

In [ ]:
KOLOM = "gabungan_2024"   # Perkotaan dan Perdesaan 2024, kolom (7) Tabel 2.5.14
prov = prov_mentah[["kode_bps", "provinsi_bps", "luas_km2", "geometry"]].merge(
    bps[["kode_bps", KOLOM, "perkotaan_2024", "perdesaan_2024"]],
    on="kode_bps", how="outer", validate="one_to_one", indicator=True)
print(prov["_merge"].value_counts().to_dict())
assert len(prov) == 38 and (prov["_merge"] == "both").all(), "Gabung belum 1:1, periksa crosswalk"
assert prov["kode_bps"].is_unique
prov = gpd.GeoDataFrame(prov.drop(columns="_merge"), geometry="geometry", crs="EPSG:4326")
print("Nilai kosong (akan tampil 'Tidak tersedia'):", int(prov[KOLOM].isna().sum()))

## Peta tematik (choropleth) PNG
Data kosong diwarnai abu-abu dengan label **“Tidak tersedia”**.

In [ ]:
BATAS = [0, 60, 80, 90, 95, 100.0001]
LABEL = ["< 60%", "60–<80%", "80–<90%", "90–<95%", "≥ 95%"]
WARNA = ["#ffffcc", "#a1dab4", "#41b6c4", "#2c7fb8", "#253494"]
WARNA_NA, LABEL_NA = "#bdbdbd", "Tidak tersedia"

def kelas(v):
    if pd.isna(v): return LABEL_NA
    for b0, b1, lab in zip(BATAS[:-1], BATAS[1:], LABEL):
        if b0 <= v < b1: return lab
WARNA_KELAS = dict(zip(LABEL, WARNA)) | {LABEL_NA: WARNA_NA}
prov["kelas"] = prov[KOLOM].apply(kelas)
prov["warna"] = prov["kelas"].map(WARNA_KELAS)
prov["kelas"].value_counts().reindex(LABEL + [LABEL_NA], fill_value=0)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
prov.plot(ax=ax, color=prov["warna"], edgecolor="white", linewidth=0.5)
ax.legend(handles=[Patch(facecolor=WARNA_KELAS[k], edgecolor="#999999", label=f"{k} ({(prov['kelas'] == k).sum()})")
                   for k in LABEL + [LABEL_NA]],
          title="% rumah tangga", loc="lower left", frameon=False, fontsize=10)
ax.set_title("Rumah tangga yang menempati rumah dengan ketahanan bangunan memenuhi syarat, 2024", loc="left")
ax.annotate("Sumber: BPS, Statistik Lingkungan Hidup Indonesia 2025, Tabel 2.5.14 (Susenas 2024, perkotaan + perdesaan). "
            "Batas: GeoJSON publik 38 provinsi.", (0, -0.04), xycoords="axes fraction", fontsize=8, color="#555555")
ax.set_axis_off()
plt.savefig("output/peta_choropleth_ketahanan_bangunan_2024.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# Peringkat untuk dibaca bersama peta
n_terendah = 5
prov.sort_values(KOLOM)[["provinsi_bps", KOLOM, "perkotaan_2024", "perdesaan_2024"]].head(n_terendah)

In [ ]:
BASEMAP = "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
m = folium.Map(location=[-2.5, 118], zoom_start=5, tiles=BASEMAP, attr="Tiles &copy; Esri")
lapisan = prov[["kode_bps", "provinsi_bps", KOLOM, "kelas", "warna", "geometry"]].copy()
lapisan["nilai_teks"] = lapisan[KOLOM].map(lambda v: LABEL_NA if pd.isna(v) else f"{v:.2f}%".replace(".", ","))
folium.GeoJson(
    lapisan.to_json(), name="Ketahanan bangunan 2024",
    style_function=lambda f: {"fillColor": f["properties"]["warna"], "color": "white", "weight": 0.7, "fillOpacity": 0.85},
    highlight_function=lambda f: {"weight": 2, "color": "#333333"},
    tooltip=folium.GeoJsonTooltip(["provinsi_bps", "nilai_teks", "kelas"], aliases=["Provinsi", "Persentase 2024", "Kelas"]),
).add_to(m)
baris = "".join(f'<div><span style="display:inline-block;width:14px;height:14px;background:{WARNA_KELAS[k]};'
                f'border:1px solid #999;margin-right:6px"></span>{k} ({(prov["kelas"] == k).sum()})</div>'
                for k in LABEL + [LABEL_NA])
legenda = (f'<div id="legenda" style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:10px 12px;'
           f'border:1px solid #999;font:13px sans-serif"><b>% rumah tangga</b>{baris}'
           f'<div style="margin-top:6px;color:#555;font-size:11px">BPS, SLHI 2025, Tabel 2.5.14 (Susenas 2024)</div></div>')
m.get_root().html.add_child(folium.Element(legenda))
folium.LayerControl().add_to(m)
m.save("output/peta_choropleth_ketahanan_bangunan_2024.html")
m

In [ ]:
# Cek: nilai di HTML = nilai sumber BPS
html_geo = json.loads(lapisan.to_json())
nilai_html = {f["properties"]["kode_bps"]: f["properties"][KOLOM] for f in html_geo["features"]}
beda = [k for k, v in nilai_html.items() if not (pd.isna(v) and pd.isna(bps.set_index("kode_bps").loc[k, KOLOM]))
        and v != bps.set_index("kode_bps").loc[k, KOLOM]]
print("Poligon di HTML:", len(nilai_html), "| nilai berbeda dari sumber:", len(beda))

## Peta persebaran episentrum USGS dan validasinya
Setiap titik punya **ID unik USGS** dan **koordinat asli**. Titik yang gagal validasi dikeluarkan dan alasannya dicatat. Provinsi hanya diberikan bila episentrum **berada di dalam** poligon (`within`). Tidak ada radius: titik di laut tetap berlabel “di luar poligon provinsi”.

In [ ]:
AWAL, AKHIR = pd.Timestamp("2026-01-01", tz="UTC"), pd.Timestamp("2026-11-17", tz="UTC")
usgs["waktu_utc"] = pd.to_datetime(usgs["time"], utc=True, errors="coerce")
alasan = validasi_koordinat(usgs, akurasi="horizontalError")
aturan_tambahan = [
    ("id kosong", usgs["id"].isna()),
    ("id duplikat", usgs["id"].duplicated(keep="first")),
    ("di luar kotak studi", ~usgs["latitude"].between(-11.5, 6.5) | ~usgs["longitude"].between(94, 142)),
    ("waktu di luar periode", usgs["waktu_utc"].isna() | (usgs["waktu_utc"] < AWAL) | (usgs["waktu_utc"] >= AKHIR)),
    ("bukan gempa bumi", usgs["type"] != "earthquake"),
    ("magnitudo < 4,5 atau kosong", usgs["mag"].isna() | (usgs["mag"] < 4.5)),
]
for nama, gagal in aturan_tambahan:
    alasan[gagal & (alasan == "")] = nama
dikeluarkan = usgs[alasan != ""].assign(alasan=alasan[alasan != ""])
dikeluarkan[["id", "time", "latitude", "longitude", "mag", "alasan"]].to_csv("output/episentrum_dikeluarkan.csv", index=False)
valid = usgs[alasan == ""]
print(f"Periode data: {valid['waktu_utc'].min():%d %b %Y %H:%M} – {valid['waktu_utc'].max():%d %b %Y %H:%M} UTC")
print("Valid:", len(valid), "| dikeluarkan:", len(dikeluarkan), "| alasan:", dikeluarkan["alasan"].value_counts().to_dict())

In [ ]:
titik = gpd.GeoDataFrame(valid, geometry=gpd.points_from_xy(valid["longitude"], valid["latitude"]), crs="EPSG:4326")
predikat = "within"
titik = gpd.sjoin(titik, prov[["kode_bps", "provinsi_bps", "geometry"]], how="left", predicate=predikat).drop(columns="index_right")
assert titik["id"].is_unique
titik["provinsi_bps"] = titik["provinsi_bps"].fillna("di luar poligon provinsi")
print(titik["provinsi_bps"].eq("di luar poligon provinsi").sum(), "episentrum di laut/negara lain;",
      titik["kode_bps"].notna().sum(), "di dalam poligon provinsi")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
prov.plot(ax=ax, color="#eeeeee", edgecolor="#bbbbbb", linewidth=0.4)
sc = ax.scatter(titik["longitude"], titik["latitude"], s=(titik["mag"] - 4) ** 3 * 8, c=titik["depth"],
                cmap="viridis_r", vmax=300, alpha=0.6, edgecolor="none")
plt.colorbar(sc, ax=ax, shrink=0.6, label="kedalaman (km)")
ax.set_title(f"{len(titik)} episentrum M≥4,5, 1 Jan–16 Nov 2026 (UTC), ukuran = magnitudo", loc="left")
ax.annotate("Sumber: katalog USGS (ID unik, koordinat asli). Kotak studi 94–142° BT, 11,5° LS–6,5° LU.",
            (0, -0.04), xycoords="axes fraction", fontsize=8, color="#555555")
ax.set_axis_off()
plt.savefig("output/peta_episentrum_usgs_2026.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

## Pola dan perbedaan antarwilayah: Moran's I dan LISA
Tetangga = 4 provinsi terdekat (KNN), karena banyak provinsi berupa pulau tanpa batas darat.

In [ ]:
# Perbedaan antarwilayah: peringkat 38 provinsi terhadap angka nasional 84,71%
NASIONAL = 84.71   # baris Indonesia, kolom (7) 2024, Tabel 2.5.14
urut = prov.sort_values(KOLOM)
fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(urut["provinsi_bps"], urut[KOLOM], color=urut["warna"], edgecolor="#888888", linewidth=0.3)
ax.axvline(NASIONAL, color="#c8401c", lw=1.5); ax.text(NASIONAL + 0.5, -1.2, f"Indonesia {NASIONAL:.2f}%".replace(".", ","), color="#c8401c")
ax.set_xlabel("% rumah tangga, 2024"); ax.tick_params(axis="y", labelsize=8)
plt.savefig("output/peringkat_ketahanan_bangunan_2024.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()
print("Rentang:", urut[KOLOM].min(), "–", urut[KOLOM].max(), "| di bawah nasional:", int((prov[KOLOM] < NASIONAL).sum()), "provinsi")

In [ ]:
np.random.seed(42)
pusat = prov.to_crs("EPSG:6933"); pusat["geometry"] = pusat.centroid
ada = prov[KOLOM].notna().values           # Moran hanya untuk provinsi yang bernilai
k = 4
w = KNN.from_dataframe(pusat[ada], k=k); w.transform = "r"
y = prov.loc[ada, KOLOM].values
mi = Moran(y, w, permutations=999)
print(f"Moran's I = {mi.I:.3f} | E(I) = {mi.EI:.3f} | p (999 permutasi) = {mi.p_sim:.3f}")

In [ ]:
lisa = Moran_Local(y, w, permutations=999, seed=42)
lab = np.array(["Tidak signifikan", "High-High", "Low-High", "Low-Low", "High-Low"])
prov["lisa"] = LABEL_NA
prov.loc[ada, "lisa"] = lab[np.where(lisa.p_sim < 0.05, lisa.q, 0)]
WL = {"High-High": "#253494", "Low-Low": "#e6550d", "Low-High": "#fdae6b", "High-Low": "#74a9cf",
      "Tidak signifikan": "#eeeeee", LABEL_NA: WARNA_NA}
fig, ax = plt.subplots(figsize=(14, 6))
prov.plot(ax=ax, color=prov["lisa"].map(WL), edgecolor="white", linewidth=0.5)
ax.legend(handles=[Patch(color=c, label=f"{k} ({(prov['lisa'] == k).sum()})") for k, c in WL.items() if (prov["lisa"] == k).any()],
          loc="lower left", frameon=False)
ax.set_title("LISA ketahanan bangunan rumah 2024 (KNN k=4, α = 0,05)", loc="left"); ax.set_axis_off()
plt.savefig("output/peta_lisa_ketahanan_bangunan_2024.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()
prov.loc[prov["lisa"].isin(["High-High", "Low-Low", "Low-High", "High-Low"]), ["provinsi_bps", KOLOM, "lisa"]].sort_values("lisa")

Pertanyaan: apakah provinsi dengan persentase rendah cenderung bertetangga? Sebutkan klaster yang terlihat.

**Jawaban:** Moran's I sekitar 0,09 dengan p sekitar 0,12 (k=4): tidak signifikan pada α = 0,05. Provinsi dengan persentase rendah tidak mengelompok, tetapi tersebar: Kepulauan Bangka Belitung, Kepulauan Riau, dan DKI Jakarta di barat; NTT, Papua Tengah, dan Papua Pegunungan di timur. LISA hanya menemukan klaster High-High kecil (Sumatera Utara, Kalimantan Utara, Maluku Utara). Jadi perbedaan antarwilayahnya besar (40,85% sampai 99,39%), tetapi penyebabnya lebih mungkin kondisi lokal tiap provinsi daripada efek tetangga. Dengan k=3 nilai p mendekati 0,06, jadi kesimpulan ini perlu disebut sensitif terhadap W. Nilai p bisa sedikit berbeda antar-run.

## Dari peta ke keputusan
Gabungkan dua informasi tanpa radius: persentase ketahanan bangunan (Susenas) dan jumlah episentrum yang **berada di dalam** poligon provinsi.

In [ ]:
prov["episentrum_di_dalam"] = prov["kode_bps"].map(titik.groupby("kode_bps").size()).fillna(0).astype(int)
prioritas = prov[(prov[KOLOM] < 80) & (prov["episentrum_di_dalam"] > 0)]
prov.sort_values(KOLOM)[["provinsi_bps", KOLOM, "episentrum_di_dalam"]].head(8)

In [ ]:
print("Provinsi < 80% yang juga punya episentrum M≥4,5 di dalam poligonnya:", prioritas["provinsi_bps"].tolist())

1. **Temuan** (dengan angka dan sumber).
2. **Batasan** (definisi indikator BPS, data episentrum hanya M≥4,5 dan hanya yang di dalam poligon, batas wilayah publik, ketidakpastian lokasi ±9 km).
3. **Rekomendasi** (wilayah prioritas dan data tambahan yang dibutuhkan).

**Jawaban:** (1) Persentase rumah tangga dengan ketahanan bangunan memenuhi syarat 2024 terendah di Papua Pegunungan (40,85%), Kepulauan Bangka Belitung (41,99%), dan DKI Jakarta (45,26%), jauh di bawah angka nasional 84,71% (BPS, SLHI 2025, Tabel 2.5.14). <br>
(2) Indikator ini tentang syarat atap, dinding, dan lantai menurut definisi BPS, bukan ketahanan gempa; episentrum hanya dihitung bila berada di dalam poligon provinsi. <br>
(3) Provinsi yang rendah persentasenya sekaligus punya episentrum di daratannya, sesuai daftar di sel atas, layak diprioritaskan untuk kajian lanjutan rumah tahan gempa, dilengkapi data jumlah rumah dan jenis bangunan.

In [ ]:
prov.drop(columns="geometry").to_csv("output/tabel_provinsi_hasil.csv", index=False)
print(sorted(os.listdir("output")))